# NB 04: Basis Risk Analysis

Rolling price correlation, price spread statistics, basis VaR/CVaR,
tail risk during volatility events, and liquidation risk simulations.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from src.config import EXCHANGES, ASSETS, RAW_DIR, PROCESSED_DIR, FIGURES_DIR
from src.models.basis_risk import (
    price_spread_stats,
    rolling_correlation,
    basis_var,
    comprehensive_basis_analysis,
    liquidation_margin_simulation,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

## 1. Load Price Data

In [ ]:
# Load available price data
price_data = {}
for exchange_dir in (RAW_DIR / 'prices').iterdir():
    if not exchange_dir.is_dir():
        continue
    exchange = exchange_dir.name
    for f in exchange_dir.glob('*.parquet'):
        asset = f.stem
        df = pd.read_parquet(f)
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
        df = df.set_index('timestamp').sort_index()
        price_data[(exchange, asset)] = df

print(f'Loaded price data for {len(price_data)} exchange-asset pairs:')
for (ex, asset), df in sorted(price_data.items()):
    print(f'  {ex}/{asset}: {len(df)} candles, {df.index.min().date()} to {df.index.max().date()}')

# Also load Drift/dYdX oracle prices from funding rate data
for dex in ['drift', 'dydx']:
    dex_dir = RAW_DIR / 'funding_rates' / dex
    if not dex_dir.exists():
        continue
    for f in dex_dir.glob('*.parquet'):
        asset = f.stem
        df = pd.read_parquet(f)
        if 'oracle_price_twap' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
            df = df.set_index('timestamp')[['oracle_price_twap']].rename(
                columns={'oracle_price_twap': 'close'}
            ).sort_index()
            price_data[(dex, asset)] = df
        elif 'price' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
            df = df.set_index('timestamp')[['price']].rename(
                columns={'price': 'close'}
            ).sort_index()
            price_data[(dex, asset)] = df

print(f'\nTotal price series (incl. oracle prices): {len(price_data)}')

## 2. Cross-Exchange Price Spread

In [ ]:
spread_results = []

for asset in ASSETS:
    exchanges_with_prices = [ex for (ex, a) in price_data if a == asset]
    
    for ex_a, ex_b in combinations(sorted(set(exchanges_with_prices)), 2):
        pa = price_data.get((ex_a, asset))
        pb = price_data.get((ex_b, asset))
        if pa is None or pb is None:
            continue
        
        # Align to common timestamps (resample to 1h)
        pa_1h = pa['close'].resample('1h').last().dropna()
        pb_1h = pb['close'].resample('1h').last().dropna()
        common = pa_1h.index.intersection(pb_1h.index)
        
        if len(common) < 100:
            continue
        
        stats_dict = price_spread_stats(pa_1h.loc[common], pb_1h.loc[common])
        stats_dict['exchange_a'] = ex_a
        stats_dict['exchange_b'] = ex_b
        stats_dict['asset'] = asset
        stats_dict['n_obs'] = len(common)
        spread_results.append(stats_dict)

spread_df = pd.DataFrame(spread_results)
if not spread_df.empty:
    print('Price spread statistics (bps):')
    spread_df[['exchange_a', 'exchange_b', 'asset', 'mean_spread_bps',
               'std_spread_bps', 'max_spread_bps', 'correlation']].round(2)

## 3. Basis VaR (95% / 99%)

In [ ]:
var_records = []

for asset in ASSETS:
    exchanges_with_prices = sorted(set(ex for (ex, a) in price_data if a == asset))
    
    for ex_a, ex_b in combinations(exchanges_with_prices, 2):
        pa = price_data.get((ex_a, asset))
        pb = price_data.get((ex_b, asset))
        if pa is None or pb is None:
            continue
        
        pa_1h = pa['close'].resample('1h').last().dropna()
        pb_1h = pb['close'].resample('1h').last().dropna()
        common = pa_1h.index.intersection(pb_1h.index)
        if len(common) < 100:
            continue
        
        mid = (pa_1h.loc[common] + pb_1h.loc[common]) / 2
        spread_bps = (pa_1h.loc[common] - pb_1h.loc[common]) / mid * 10_000
        
        for hp in [1, 8]:  # 1h and 8h holding
            v95 = basis_var(spread_bps, 0.95, hp)
            v99 = basis_var(spread_bps, 0.99, hp)
            
            var_records.append({
                'exchange_a': ex_a, 'exchange_b': ex_b, 'asset': asset,
                'holding_hours': hp,
                'var_95_bps': v95, 'var_99_bps': v99,
            })

var_df = pd.DataFrame(var_records)
if not var_df.empty:
    print('Basis VaR by holding period:')
    var_df.round(2)

## 4. Liquidation Risk Simulation

In [ ]:
liq_records = []

for asset in ['BTC', 'ETH', 'SOL']:
    # Use first available price source
    price_series = None
    for ex in ['binance', 'okx', 'bybit', 'bitmex']:
        if (ex, asset) in price_data:
            price_series = price_data[(ex, asset)]['close']
            break
    
    if price_series is None or len(price_series) < 100:
        continue
    
    for leverage in [1, 2, 3, 5, 10]:
        sim = liquidation_margin_simulation(price_series, leverage=leverage)
        sim['asset'] = asset
        liq_records.append(sim)

liq_df = pd.DataFrame(liq_records)
if not liq_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    for asset in liq_df['asset'].unique():
        ad = liq_df[liq_df['asset'] == asset]
        ax.plot(ad['leverage'], ad['liquidation_frequency'] * 100,
                marker='o', label=asset)
    
    ax.set_xlabel('Leverage')
    ax.set_ylabel('Liquidation Frequency (%)')
    ax.set_title('Single-Period Liquidation Risk by Leverage')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'liquidation_risk.pdf', bbox_inches='tight')
    plt.show()
    
    liq_df[['asset', 'leverage', 'liquidation_frequency',
            'close_call_frequency', 'worst_margin', 'margin_at_risk_99']].round(4)